In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("Diabetes_prediction.csv")

In [ ]:
df.head()

In [ ]:
print("Shape:", df.shape)
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
feature_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
                "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
X = df[feature_cols]
y = df["Diagnosis"]

print("Target distribution:")
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print("Feature scaling applied (StandardScaler)")

In [ ]:
perceptron = Perceptron(max_iter=1000, random_state=42, tol=1e-3)
perceptron.fit(X_train_scaled, y_train)

y_pred = perceptron.predict(X_test_scaled)
print("Perceptron model trained.")
print("Accuracy:", accuracy_score(y_test, y_pred) * 100, "%")

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Non-Diabetic", "Diabetic"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Diabetes", "Diabetes"]
).plot(cmap="Oranges")
plt.title("Perceptron Confusion Matrix")
plt.show()

In [ ]:
print("Effect of max_iter on Perceptron accuracy:")
for iters in [10, 50, 100, 500, 1000]:
    p = Perceptron(max_iter=iters, random_state=42, tol=1e-3)
    p.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, p.predict(X_test_scaled))
    print(f"  max_iter={iters:>5}  |  Accuracy: {acc*100:.2f}%")

In [ ]:
weights_df = pd.DataFrame({
    "Feature": feature_cols,
    "Weight": perceptron.coef_[0]
}).sort_values("Weight", key=abs, ascending=False)

print("Perceptron Feature Weights (absolute value sorted):")
print(weights_df.to_string(index=False))

plt.figure(figsize=(8, 5))
colors = ["green" if w > 0 else "red" for w in weights_df["Weight"]]
plt.barh(weights_df["Feature"], weights_df["Weight"], color=colors)
plt.axvline(x=0, color="black", linewidth=0.8)
plt.xlabel("Weight Value")
plt.title("Perceptron Feature Weights")
plt.tight_layout()
plt.show()